In [11]:
%pip install OpenAI

     |████████████████████████████████| 1.4 MB 5.3 MB/s eta 0:00:01
     |████████████████████████████████| 311 kB 9.6 MB/s eta 0:00:01
You should consider upgrading via the '/Users/seoyeonkim/customs-rag/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import chromadb
import torch
import json

from openai import OpenAI
from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction
)

In [ ]:
client_openai = OpenAI(
    api_key=""
)

In [14]:
client = chromadb.PersistentClient(
    path="../chroma_db"
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else (
        "mps"
        if torch.backends.mps.is_available()
        else "cpu"
    )
)

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device=device,
    normalize_embeddings=True,
)

collection = client.get_collection(
    name="customs_knowledge_v3",
    embedding_function=embedding_fn
)

print(collection.count())

6328


In [15]:
#검색 함수

METRIC = "l2"

def distance_to_similarity(distance, metric="l2"):
    if metric == "cosine":
        return max(0.0, 1 - distance)

    return max(
        0.0,
        1 - (distance ** 2) / 2
    )


def retrieve_evidence(
    query,
    where=None,
    n_results=5
):

    result = collection.query(
        query_texts=[query],
        where=where,
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ],
    )

    evidence = []

    for i, (
        doc,
        meta,
        dist
    ) in enumerate(
        zip(
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0]
        )
    ):

        evidence.append({
            "idx": i,
            "text": doc,
            "metadata": meta,
            "similarity_pct":
                round(
                    distance_to_similarity(
                        dist,
                        METRIC
                    ) * 100,
                    1
                )
        })

    return evidence

In [16]:
#context 생성

def build_context_block(evidence):

    lines = []

    for e in evidence:

        source = (
            e["metadata"].get("article")
            or e["metadata"].get("source_type")
            or "출처 미상"
        )

        lines.append(
            f"[{e['idx']}]\n"
            f"출처: {source}\n"
            f"검색유사도: {e['similarity_pct']}%\n\n"
            f"{e['text']}"
        )

    return "\n\n".join(lines)

In [17]:
#XAI schema

XAI_SCHEMA = {
    "name": "xai_response",
    "schema": {
        "type": "object",
        "properties": {

            "answer": {
                "type": "string"
            },

            "reasoning": {
                "type": "string"
            },

            "llm_confidence": {
                "type": "integer"
            },

            "cited_chunk_indices": {
                "type": "array",
                "items": {
                    "type": "integer"
                }
            }

        },

        "required": [
            "answer",
            "reasoning",
            "llm_confidence",
            "cited_chunk_indices"
        ],

        "additionalProperties": False
    },

    "strict": True
}

In [18]:
# GPT 호출

def ask_llm_with_xai(
    query,
    evidence
):

    context = build_context_block(
        evidence
    )

    prompt = f"""
당신은 관세 및 FTA 전문가입니다.

아래 문서만 근거로 답변하세요.

문서:
{context}

질문:
{query}

JSON으로 답변하세요.
"""

    response = (
        client_openai.chat.completions.create(
            model="gpt-4.1-mini",

            messages=[
                {
                    "role":"user",
                    "content":prompt
                }
            ],

            response_format={
                "type":"json_schema",
                "json_schema":XAI_SCHEMA
            }
        )
    )

    return json.loads(
        response
        .choices[0]
        .message
        .content
    )

In [19]:
#confidence


def compute_confidence(
    evidence,
    llm_result
):

    cited = (
        llm_result[
            "cited_chunk_indices"
        ]
        or
        [0]
    )

    scores = [
        e["similarity_pct"]
        for e in evidence
        if e["idx"] in cited
    ]

    retrieval_score = (
        round(
            sum(scores)/len(scores),
            1
        )
        if scores
        else 0
    )

    return retrieval_score

In [20]:
#final excution func
def run_xai_rag(
    query,
    where=None
):

    evidence = retrieve_evidence(
        query,
        where=where
    )

    llm_result = ask_llm_with_xai(
        query,
        evidence
    )

    confidence = compute_confidence(
        evidence,
        llm_result
    )

    print("\n질문")
    print(query)

    print("\n답변")
    print(
        llm_result["answer"]
    )

    print("\n이유")
    print(
        llm_result["reasoning"]
    )

    print("\n근거")

    for e in evidence:

        mark = (
            "★"
            if e["idx"]
            in llm_result[
                "cited_chunk_indices"
            ]
            else " "
        )

        print(
            f"{mark} "
            f"[{e['idx']}] "
            f"{e['similarity_pct']}%"
        )

    print(
        f"\n최종 신뢰도: "
        f"{confidence}%"
    )

    return llm_result

In [21]:
# test MVP

run_xai_rag(
    "품목분류 사전심사란 무엇인가?",
    where={
        "source_type":"customs_law"
    }
)


질문
품목분류 사전심사란 무엇인가?

답변
품목분류 사전심사란 수출입하려는 자, 수출할 물품의 제조자, 또는 관세사 등이 수출입신고를 하기 전에 대통령령으로 정하는 서류를 갖추어 관세청장에게 해당 물품에 적용될 품목분류를 미리 심사하여 줄 것을 신청하는 절차를 말한다.

이유
문서 제86조에 따르면, 품목분류 사전심사는 수출입자 또는 관련 자가 수출입신고 전에 관세청장에게 서류를 제출하여 해당 물품에 적용될 품목분류를 미리 심사받는 제도를 뜻한다. 관세청장은 신청받은 품목분류를 심사하여 정해진 기간 내에 신청인에게 통지해야 하며, 신청인은 통지받은 날부터 일정 기간 내에 재심사를 신청할 수 있다. 이를 통해 품목분류를 사전에 명확히 하여 이후 수출입 과정에서 혼란을 줄일 수 있다.

근거
★ [0] 94.9%
  [1] 93.1%
  [2] 91.3%
  [3] 90.3%
  [4] 90.1%

최종 신뢰도: 94.9%


{'answer': '품목분류 사전심사란 수출입하려는 자, 수출할 물품의 제조자, 또는 관세사 등이 수출입신고를 하기 전에 대통령령으로 정하는 서류를 갖추어 관세청장에게 해당 물품에 적용될 품목분류를 미리 심사하여 줄 것을 신청하는 절차를 말한다.',
 'reasoning': '문서 제86조에 따르면, 품목분류 사전심사는 수출입자 또는 관련 자가 수출입신고 전에 관세청장에게 서류를 제출하여 해당 물품에 적용될 품목분류를 미리 심사받는 제도를 뜻한다. 관세청장은 신청받은 품목분류를 심사하여 정해진 기간 내에 신청인에게 통지해야 하며, 신청인은 통지받은 날부터 일정 기간 내에 재심사를 신청할 수 있다. 이를 통해 품목분류를 사전에 명확히 하여 이후 수출입 과정에서 혼란을 줄일 수 있다.',
 'llm_confidence': 95,
 'cited_chunk_indices': [0]}

In [ ]:
# test MVP 2

run_xai_rag(
    "품목분류 사전심사란 무엇인가?",
    where={"source_type":"customs_law"}
)

run_xai_rag(
    "과세가격 결정방법의 사전심사란?",
    where={"source_type":"customs_law"}
)

run_xai_rag(
    "원산지증명서란?",
    where={"source_type":"customs_law"}
)